# AreaSqKm_01_Notebook
Javier Parada

October 13, 2021

In [1]:
import ee
ee.Initialize()
import leafmap
#leafmap.update_package()
import geemap
#geemap.update_package()
import os

In [2]:
gaul0 = ee.FeatureCollection("FAO/GAUL/2015/level0")
gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
gaul2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")

In [3]:
Map = geemap.Map(center = (40, -100), zoom = 3)
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(T…

In [4]:
ee_class_table = """
Value	Color	Description
1	05450a	Evergreen Needleleaf Forests: dominated by evergreen conifer trees (canopy >2m). Tree cover >60%.
2	086a10	Evergreen Broadleaf Forests: dominated by evergreen broadleaf and palmate trees (canopy >2m). Tree cover >60%.
3	54a708	Deciduous Needleleaf Forests: dominated by deciduous needleleaf (larch) trees (canopy >2m). Tree cover >60%.
4	78d203	Deciduous Broadleaf Forests: dominated by deciduous broadleaf trees (canopy >2m). Tree cover >60%.
5	009900	Mixed Forests: dominated by neither deciduous nor evergreen (40-60% of each) tree type (canopy >2m). Tree cover >60%.
6	c6b044	Closed Shrublands: dominated by woody perennials (1-2m height) >60% cover.
7	dcd159	Open Shrublands: dominated by woody perennials (1-2m height) 10-60% cover.
8	dade48	Woody Savannas: tree cover 30-60% (canopy >2m).
9	fbff13	Savannas: tree cover 10-30% (canopy >2m).
10	b6ff05	Grasslands: dominated by herbaceous annuals (<2m).
11	27ff87	Permanent Wetlands: permanently inundated lands with 30-60% water cover and >10% vegetated cover.
12	c24f44	Croplands: at least 60% of area is cultivated cropland.
13	a5a5a5	Urban and Built-up Lands: at least 30% impervious surface area including building materials, asphalt and vehicles.
14	ff6d4c	Cropland/Natural Vegetation Mosaics: mosaics of small-scale cultivation 40-60% with natural tree, shrub, or herbaceous vegetation.
15	69fff8	Permanent Snow and Ice: at least 60% of area is covered by snow and ice for at least 10 months of the year.
16	f9ffa4	Barren: at least 60% of area is non-vegetated barren (sand, rock, soil) areas with less than 10% vegetation.
17	1c0dff	Water Bodies: at least 60% of area is covered by permanent water bodies.

"""
legend_dict = geemap.legend_from_ee(ee_class_table)
Map.add_legend(legend_title="MODIS Global Land Cover", legend_dict=legend_dict)

In [5]:
roi = gaul1.filter(ee.Filter.eq('ADM1_NAME', 'Kerala'))
Map.addLayer(roi, {'color': '00909F', 'fillColor': 'b5ffb4', 'width': .1}, 'Second Level Administrative Units')
Map.centerObject(roi)

In [6]:
# Get the MODIS Yearly 500m Landcover Collection
modisLandCover = ee.Image("MODIS/006/MCD12Q1/2018_01_01")
igbpLandCover = modisLandCover.select('LC_Type1')
igbpLandCoverVis = {
  'min': 1.0,
  'max': 17.0,
  'palette': [
    '05450a', '086a10', '54a708', '78d203', '009900', 'c6b044', 'dcd159',
    'dade48', 'fbff13', 'b6ff05', '27ff87', 'c24f44', 'a5a5a5', 'ff6d4c',
    '69fff8', 'f9ffa4', '1c0dff'
  ],
}
Map.addLayer(igbpLandCover.clip(roi), igbpLandCoverVis, 'IGBP Land Cover')

In [7]:
urban = igbpLandCover.eq(13)
Map.addLayer(urban.selfMask().clip(roi), {'min': 0, 'max':1, 'palette': ['grey','black']}, 'Urban')

In [8]:
merit = ee.Image("MERIT/DEM/v1_0_3").select('dem')
lowAreas = merit.lte(5)
Map.addLayer(lowAreas.selfMask().clip(roi), {'min': 0, 'max':1, 'palette': ['grey','blue']}, "Low Areas")

In [9]:
out_dir = os.path.join(os.path.expanduser('~'), 'Downloads')
nlcd_stats = os.path.join(out_dir, 'kerala.csv')  
geemap.zonal_statistics_by_group(igbpLandCover.clip(roi), 
                                 roi, 
                                 nlcd_stats, 
                                 statistics_type='SUM', 
                                 denominator=1000000, 
                                 decimal_places=2,
                                 scale = 500)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Downloads/kerala.csv
